> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — network evidence representation

PCSF is the interpretable baseline. GNN models should be treated as optional predictive layers with held-out validation.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
# Genotypes are simulated once into a 2-bit packed Zarr store (GT_packed2, 0.25 byte per call)
# that every tutorial reuses. Traits are simulated from bounded decoded blocks, so no dense
# animals x variants matrix is held in memory; GENO is the genotype source used below.
DATA=Path("vigor_tutorial_data"); DATA.mkdir(exist_ok=True)
GENO=str(DATA/"complex_n320_m1000_chr5.zarr")
sim=vigor.simulate_packed_cohort(GENO,n_samples=320,n_variants=1000,n_chromosomes=5,n_traits=8,seed=SEED,genotype_kwargs={"generations":100},trait_kwargs=cfg)
Y=sim.phenotypes.to_pandas(); VI=vigor.result_table(sim,"variant_info")
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

edges=pd.DataFrame({"source":["geneA","geneB","geneC","geneA"],"target":["geneB","geneC","geneD","geneD"],"cost":[1.,1.,1.,2.]})
prizes={"geneA":2.0,"geneD":1.5}
print(edges); print(prizes)


In [ ]:
# A small demonstration network; these names are not HS-rat gene evidence.
from scipy.sparse.csgraph import connected_components
edges=pd.DataFrame({"source":["geneA","geneB","geneC","geneD","geneE","geneA"],
                    "target":["geneB","geneC","geneD","geneE","geneF","geneF"],"cost":[1.,1.,1.,1.,1.,2.]})
prizes={"geneA":2.,"geneF":1.5}
adjacency,nodes,node_index=vigor.edge_table_to_csr(edges)
print("CSR components:",connected_components(adjacency,directed=False)[0])
result=vigor.prioritize_network(edges,prizes,method="greedy")
print("PCSF nodes:",result.nodes)
print("diffusion:",vigor.diffuse_scores(edges,prizes))
features=pd.DataFrame(np.random.default_rng(42).normal(size=(6,3)),index=nodes)
labels=pd.Series([0,1,0,1,0,1],index=nodes)
loci=pd.Series(["l1","l1","l2","l2","l3","l3"],index=nodes)
gnn=vigor.fit_heterogeneous_gnn(edges,features,labels,
    split={"held_out_loci":["l3"]},node_loci=loci,architecture="gcn",epochs=20,patience=5,random_state=42)
print("GNN:",type(gnn).__name__)


## From fine-mapped variants to a PCSF gene subnetwork

The fine-map export from tutorial 08 provides PIP prizes on `variant:` nodes and
GTF SNP→gene edges. Genes only connect through **gene–gene edges** you supply,
such as a protein interaction network, eQTL/sQTL links, or chromatin-to-gene
contacts. Each needs explicit evidence labels and costs. The chain of genes below
is a placeholder that shows the data shape; it is not evidence.

`method="pcst_fast"` is the default exact PCST/PCSF solver (`pip install
"VIGOR[network]"`). The greedy fallback returns one connected positive-gain tree
and has no optimality guarantee. Sensitivity-test `pcsf_link_cost`, prize scaling
and `num_clusters`; they drive which genes are selected.

In [ ]:
import gzip
# Pick a window around the strongest planted additive effect of the first three traits. Truth
# only chooses where to look; the fit below never sees it.
traits3=list(Y.columns[:3])
VE=vigor.result_table(sim,"variant_effects")
VE=VE[VE.trait.isin(traits3)]
lead=str(VE.loc[VE.additive_effect_final_mean.abs().idxmax(),"snp"])
chrom=str(VI.loc[lead,"chrom"]); on_chrom=VI.index[VI.chrom.astype(str)==chrom]
k=int(on_chrom.get_loc(lead))
Xw=vigor.genotype_panel(GENO,chromosome=chrom,variants=slice(max(0,k-40),k+40))  # bounded dense window
# The packed store has no allele labels: A1/A2 below are synthetic, with dosage counting A2.
vw=pd.DataFrame({"chrom":chrom,"pos":VI.loc[Xw.columns,"pos"].astype(int).to_numpy(),"A1":"A","A2":"G"},index=Xw.columns)
# Synthetic GTF: one 4-variant gene every 8 variants, tiled independently of the simulated truth,
# with a CDS over each gene's first variant. Real analyses use the assembly-matched RefSeq/Ensembl
# GTF (gtf_contig_aliases maps "12" to "NC_086030.1", annotation_assembly/variant_assembly check it).
gtf=DATA/f"synthetic_chr{chrom}_window.gtf.gz"
with gzip.open(gtf,"wt") as f:
    for g,i in enumerate(range(0,len(vw)-3,8)):
        a,b=max(1,int(vw.pos.iloc[i])-50),int(vw.pos.iloc[i+3])+50
        f.write(f'{chrom}\tsynthetic\tgene\t{a}\t{b}\t.\t+\t.\tgene_id "SG{g}"; gene "Gene{g}"; gene_biotype "protein_coding";\n')
        f.write(f'{chrom}\tsynthetic\tCDS\t{a}\t{a+200}\t.\t+\t0\tgene_id "SG{g}"; gene "Gene{g}";\n')
bfm=vigor.fine_map_region(Xw,Y,trait_subset=traits3,methods=("bayes",),L=4,hdbscan=False,
    use_evidence_priors=False,gtf_path=gtf,gtf_variant_metadata=vw,allow_unverified_assembly=True,
    pcsf_export=True,pcsf_genotype_dosage_allele="A2",pcsf_link_cost=0.05,
    bayes_kwargs={"max_iter":100,"tol":1e-5})
bayes=bfm.fits["bayes"]
export=bfm.metadata["pcsf_inputs"]
prizes=export["pcsf_prizes"]
if len(prizes)==0:
    print("No credible set passed the evidence/purity screens; nothing to prize.")
else:
    genes=sorted(set(export["pcsf_edges"].target))
    placeholder=pd.DataFrame({"source":genes[:-1],"target":genes[1:],"cost":0.5})  # stand-in gene-gene links
    edges=pd.concat([export["pcsf_edges"],placeholder],ignore_index=True)
    try:
        import pcst_fast  # noqa: F401
        method="pcst_fast"
    except ImportError:
        method="greedy"
    forest=vigor.prize_collecting_steiner_forest(edges,prizes,method=method,prize_transform="none")
    print(method,"objective",round(forest.objective,4),forest.warnings)
    print(forest.nodes)
    print("selected genes:",[n for n in forest.nodes.node if not str(n).startswith("variant:")])
    print(export["gene_ranks"].head())
